   
# 📄 ReoCosInsurance Report

Migration of on-prem `[SSRS].[ReoCosInsurance]` stored procedure to Databricks.

## 🔗 Links
- **[BRD](https://cepalhellas.atlassian.net/wiki/x/AYDJO)**


---

**Source tables:**
* `primary.insurance` — Insurance contracts per real estate
* `primary.realestatebuilding` — Property details (filtered to `SourceSystem = 'thesis'`)
* `primary.realestateinfo` — Ownership records
* `primary.party` — Party/company details (filtered to `SourceSystem = 'thesis'`)
* `landed.thesis_prproperty` — Insurance category lookup

**Output table:** `{env}-cdh-delivery.internalreports.reo_cos_insurance`

## Environment Setup

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared"

def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None

def get_scope_key_names(secrets_scope):
    try:
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)
aws_key = aws_keys[0] if aws_keys else None
aws_secret_key = get_aws_credentials(secrets_scope, aws_key)

if mode == "shared":
    print("mode is shared")
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                              aws_access_key_id=aws_key,
                              aws_secret_access_key=aws_secret_key)
else:
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

def connect_to_aws_and_access_ssm(ssm_key):
    if aws_secret_key is None:
        print("AWS secret key not available.")
        return
    response = ssm_client.get_parameter(Name=ssm_key, WithDecryption=False)
    return response['Parameter']['Value']

env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

## Input Parameters

**run_date:** The date value to run the report. When called by the orchestrator, an empty `run_date` means "use today's date" (Athens timezone). The `BusinessDate` is then resolved as the latest available date in the insurance table up to the `run_date`.

**mode:** Cluster access mode (`shared` or `single`). Defaults to `shared` if not set.

In [0]:
from datetime import datetime
from pytz import timezone

# Environment — resolved from SSM (cell above) or defaults to 'p'
if env_prefix is not None:
    env = env_prefix
else:
    env = 'p'

# run_date — from widget (orchestrator) or defaults to today (Athens TZ)
athens_tz = timezone('Europe/Athens')
try:
    run_date = dbutils.widgets.get("run_date")
    if run_date is None or run_date == "":
        raise ValueError("Empty run_date")
except Exception as e:
    run_date = datetime.now(athens_tz).strftime('%Y-%m-%d')

referenceDate = run_date

# BusinessDate — latest available up to the referenceDate
Businessdate = spark.sql(f"""
    SELECT MAX(BusinessDate) AS BusinessDate
    FROM `{env}-cdh`.primary.insurance
    WHERE BusinessDate <= '{referenceDate}'
""").collect()[0]["BusinessDate"]
Businessdate = str(Businessdate)

print(f"BusinessDate: {Businessdate}")

In [0]:
# Source tables (Unity Catalog)
TBL_INSURANCE       = f"`{env}-cdh`.primary.insurance"
TBL_REALESTATE      = f"`{env}-cdh`.primary.realestatebuilding"
TBL_REALESTATEINFO  = f"`{env}-cdh`.primary.realestateinfo"
TBL_PARTY           = f"`{env}-cdh`.primary.party"
TBL_PRPROPERTY      = f"`{env}-cdh`.landed.thesis_prproperty"

# Delivery table
DELIVERY_CATALOG    = f"`{env}-cdh-delivery`"
DELIVERY_SCHEMA     = "internalreports"
DELIVERY_TABLE      = "reo_cos_insurance"
DELIVERY_PATH       = f"{DELIVERY_CATALOG}.{DELIVERY_SCHEMA}.{DELIVERY_TABLE}"

print(f"""Tables configured (env={env}):
  Insurance       → {TBL_INSURANCE}
  RealEstate      → {TBL_REALESTATE}
  RealEstateInfo  → {TBL_REALESTATEINFO}
  Party           → {TBL_PARTY}
  PrProperty      → {TBL_PRPROPERTY}
  Delivery        → {DELIVERY_PATH}
  BusinessDate    → {Businessdate}""")

## Step 1: `insurance_1` — Insurance count per property
Counts distinct insurance contracts and determines group vs individual insurance flag per property.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW insurance_1 AS
SELECT 
    COUNT(DISTINCT COALESCE(CAST(ins.GroupInsuranceFlag AS INT), 0)) AS insurance_type,
    ins.RealEstateKey,
    COUNT(DISTINCT ins.InsuranceId) AS insurance_count,
    MAX(CAST(ins.GroupInsuranceFlag AS INT)) AS group_insurance
FROM {TBL_INSURANCE} ins
WHERE ins.BusinessDate = '{Businessdate}'
GROUP BY ins.RealEstateKey
""")

## Step 2: `insurance_2` — Latest insurance per property
Selects the most recent insurance record (by end date, then start date) per property, including company name.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW insurance_2 AS
SELECT * FROM (
    SELECT 
        ROW_NUMBER() OVER(
            PARTITION BY ins.RealEstateKey 
            ORDER BY ins.InsuranceEndDate DESC,
                     ins.InsuranceStartDate DESC
        ) AS rnk,
        ins.InsuranceStartDate,
        ins.InsuranceEndDate,
        party.LastNameGR AS Company,
        ins.InsuranceType AS InsuranceTypeDescription,
        ins.RealEstateKey
    FROM {TBL_INSURANCE} ins
    LEFT JOIN {TBL_PARTY} party
        ON party.PartyKey = ins.CompanyKey
        AND party.BusinessDate = '{Businessdate}'
        AND party.PartyType = 'Organization'
        AND party.PartyRepresentKey LIKE '%INSUR%'
    WHERE ins.BusinessDate = '{Businessdate}'
) AS a WHERE a.rnk = 1
""")

## Step 3: `pivot_base` — Prepare data for contract pivot
Assigns row numbers per property (ordered by end date desc) for flattening into contract columns.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW pivot_base AS
SELECT 
    ins.RealEstateKey,
    ROW_NUMBER() OVER(
        PARTITION BY ins.RealEstateKey 
        ORDER BY ins.InsuranceEndDate DESC, ins.InsuranceStartDate DESC
    ) AS rn,
    ins.AgreementNumber AS contractno,
    ins.InsuranceStartDate AS OpenDate,
    ins.InsuranceEndDate AS ExpiryDate
FROM {TBL_INSURANCE} ins
WHERE ins.BusinessDate = '{Businessdate}'
""")

## Step 4: `contract_insurance` — Pivot contracts into columns
Flattens top 3 insurance contracts per property into columns: Contract 1/2/3 with Start/End dates.

In [0]:
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW contract_insurance AS
SELECT 
    RealEstateKey,
    MAX(CASE WHEN rn = 1 THEN contractno END) AS `Contract 1`,
    MAX(CASE WHEN rn = 1 THEN OpenDate END)   AS `Start Date 1`,
    MAX(CASE WHEN rn = 1 THEN ExpiryDate END) AS `End Date 1`,
    MAX(CASE WHEN rn = 2 THEN contractno END) AS `Contract 2`,
    MAX(CASE WHEN rn = 2 THEN OpenDate END)   AS `Start Date 2`,
    MAX(CASE WHEN rn = 2 THEN ExpiryDate END) AS `End Date 2`,
    MAX(CASE WHEN rn = 3 THEN contractno END) AS `Contract 3`,
    MAX(CASE WHEN rn = 3 THEN OpenDate END)   AS `Start Date 3`,
    MAX(CASE WHEN rn = 3 THEN ExpiryDate END) AS `End Date 3`
FROM pivot_base
WHERE rn <= 3
GROUP BY RealEstateKey
""")

## Step 5: `data_main` — Main property + insurance query

Joins ownership (`realestateinfo`) with property details (`realestatebuilding`), owner names (`party`), and all insurance temp views into a single denormalized result.

**Key CTEs:**

| CTE | Purpose |
| --- | --- |
| `re_dedup` | `SELECT DISTINCT` on `realestatebuilding` — removes pure duplicate rows (33K properties affected) |
| `party_dedup` | Filters `SourceSystem = 'thesis'` to match on-prem `dwh.prt.Party`. Builds `OwnerName` from `FirstNameGR` + `LastNameGR`, with org-name duplication fix (`STARTSWITH` check) |
| `rei_dedup` | `ROW_NUMBER()` per `(RealEstateKey, OwnerPartyKey)` — collapses ownership×lien explosion to 1 row per (property, owner), matching on-prem `PropertyOwnership` granularity |
| `owner_dedup` | `ROW_NUMBER()` per `(PropertyThesisID, OwnerName)` — collapses same-name owners with different `OwnerPartyKeys` (e.g. bare ownership vs usufruct). Priority: Πλήρης κυριότητα > Ψιλή κυριότητα > Επικαρπία > Άγνωστη |

**Filters:**
* `SourceSystem = 'thesis'` on `realestatebuilding` and `party`
* `OwnershipStatusCode = 'Active'` — equivalent to on-prem `OwnershipStatusId IN (11006, 11008)`. The N/A (11008) records from on-prem’s `ISNULL(…, 'N/A')` transformation are not reproducible on cloud (0 NULL records in thesis source).
* `OwnershipPercentage > 0`
* PersonId filter removed (dead code in on-prem SP — returned 0 rows)

**Known limitations (not fixable without upstream changes):**
* **Name truncation:** `LastNameGR` caps at \~50 chars; `LegalNameGR` is NULL for all orgs. Affects \~2,796 records.
* **Spacing:** On-prem `PartyName` had inconsistent single/double spacing (83%/17%). Cloud always uses double space. Affects \~1,788 records.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW data_main AS
WITH
-- Pre-deduplicate realestatebuilding (33K properties have pure duplicate rows)
re_dedup AS (
    SELECT DISTINCT
        RealEstateKey, RealEstateCepalKey, RealEstateStatus,
        RealEstateCategory, RealEstateSubCategory, RealEstateDescription,
        RealEstateAddress1, RealEstateArea, RealEstateCity, RealEstatePostalCode,
        RealEstateLandArea, RealEstateNetArea, RealEstateConstructionYear,
        RealEstateMarketValue
    FROM {TBL_REALESTATE}
    WHERE BusinessDate = '{Businessdate}'
      AND SourceSystem = 'thesis'
),
-- =====================================================================
-- Party dedup: thesis source system only (matches on-prem dwh.prt.Party)
-- =====================================================================
-- Name logic:
--   • NULL/empty FirstNameGR (orgs) → just LastNameGR
--   • Organization where LastNameGR starts with FirstNameGR → just LastNameGR
--     (avoids duplication like "REOCO - GALAXY II  REOCO - GALAXY II (SPV 63)")
--   • Individuals → CONCAT(FirstNameGR, '  ', LastNameGR)  (double space matches
--     on-prem dwh.prt.Party.PartyName for the majority of records)
-- =====================================================================
party_dedup AS (
    SELECT DISTINCT
        PartyKey,
        CASE
            WHEN FirstNameGR IS NULL OR TRIM(FirstNameGR) = '' THEN TRIM(LastNameGR)
            WHEN PartyType = 'Organization' AND STARTSWITH(TRIM(LastNameGR), TRIM(FirstNameGR))
            THEN TRIM(LastNameGR)
            ELSE TRIM(CONCAT_WS('  ', FirstNameGR, LastNameGR))
        END AS OwnerName
    FROM {TBL_PARTY}
    WHERE BusinessDate = '{Businessdate}'
      AND SourceSystem = 'thesis'
),
-- =====================================================================
-- rei_dedup: Deduplicate realestateinfo to ONE row per (property, owner)
-- =====================================================================
-- WHY: On-prem uses dwh.ast.PropertyOwnership which stores one row per
-- ownership record. Cloud's realestateinfo is denormalized — it merges
-- ownership data with lien data (RELienKey, RELienReferenceKey, etc.),
-- creating one row per (ownership × lien) combination. Each such row has
-- a unique RealEstateInfoKey (e.g. 338604;125009, 338604;125008).
--
-- Without this dedup, DWHPropertyID = RealEstateInfoKey is unique per
-- lien, so SELECT DISTINCT cannot collapse rows — e.g. property 249546
-- produced 150 rows (5 owners × 30 liens each) vs 5 rows on-prem.
--
-- ROW_NUMBER partitioned by (RealEstateKey, OwnerPartyKey) picks one
-- representative row per (property, owner), matching PropertyOwnership
-- granularity. The chosen RealEstateInfoKey is arbitrary but acceptable
-- since DWHPropertyID doesn't match on-prem anyway (different key systems).
-- =====================================================================
rei_dedup AS (
    SELECT *
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY RealEstateKey, OwnerPartyKey
                ORDER BY RealEstateInfoKey
            ) AS rn
        FROM {TBL_REALESTATEINFO}
        WHERE BusinessDate = '{Businessdate}'
          AND OwnershipStatusCode = 'Active'
          AND OwnershipPercentage > 0
    )
    WHERE rn = 1
),
-- Core property-ownership join (one row per ownership record)
total AS (
    SELECT
        re.RealEstateKey AS PropertyThesisID,
        re.RealEstateStatus AS PropertyStatus,
        re.RealEstateCepalKey AS PropertyId,
        rei.RealEstateInfoKey AS DWHPropertyID,
        re.RealEstateCategory AS PropertyCategory,
        re.RealEstateSubCategory AS PropertySubCategory,
        re.RealEstateDescription AS PropertyDescription,
        re.RealEstateAddress1 AS Address,
        re.RealEstateArea AS Area,
        re.RealEstateCity AS City,
        re.RealEstatePostalCode AS ZipCode,
        re.RealEstateLandArea AS LandArea,
        re.RealEstateNetArea AS NetArea,
        re.RealEstateConstructionYear AS ContructionYear,
        re.RealEstateMarketValue AS MarketValue,
        party.OwnerName,
        rei.OwnershipTypeCode     -- for owner_dedup priority
    FROM rei_dedup rei
    INNER JOIN re_dedup re
        ON re.RealEstateKey = rei.RealEstateKey
    LEFT JOIN party_dedup party
        ON party.PartyKey = rei.OwnerPartyKey
),
-- =====================================================================
-- owner_dedup: Collapse duplicate OwnerNames per property
-- =====================================================================
-- The thesis system records the same physical person under multiple
-- OwnerPartyKeys (e.g. bare ownership, usufruct, full ownership).
-- After joining to party_dedup, these all resolve to the same OwnerName
-- but with different DWHPropertyID/OwnershipTypeCode.
-- On-prem had strictly 1 row per (PropertyThesisID, OwnerName).
--
-- 98% of duplicates have identical type & percentage — choice is
-- irrelevant. For the ~1% with different types, we prefer:
-- Priority: Πλήρης κυριότητα > Ψιλή κυριότητα > Επικαρπία > Άγνωστη
-- =====================================================================
owner_dedup AS (
    SELECT PropertyThesisID, PropertyStatus, PropertyId, DWHPropertyID,
           PropertyCategory, PropertySubCategory, PropertyDescription,
           Address, Area, City, ZipCode,
           LandArea, NetArea, ContructionYear, MarketValue,
           OwnerName
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY PropertyThesisID, OwnerName
                ORDER BY CASE OwnershipTypeCode
                    WHEN 'Πλήρης κυριότητα' THEN 1
                    WHEN 'Ψιλή κυριότητα' THEN 2
                    WHEN 'Επικαρπία' THEN 3
                    WHEN 'Άγνωστη' THEN 4
                    ELSE 5
                END
            ) AS owner_rn
        FROM total
    ) WHERE owner_rn = 1
)
SELECT 
    owner_dedup.*,
    CASE WHEN ins2.InsuranceEndDate > CURRENT_DATE() THEN 'Yes' ELSE 'No' END AS InsuranceFlag,
    ci.`Contract 1`,  ci.`Start Date 1`,  ci.`End Date 1`,
    ci.`Contract 2`,  ci.`Start Date 2`,  ci.`End Date 2`,
    ci.`Contract 3`,  ci.`Start Date 3`,  ci.`End Date 3`,
    ins2.Company AS InsuranceCompany,
    ins2.InsuranceStartDate AS InsuranceDateStart,
    ins2.InsuranceEndDate AS InsuranceDateEnd,
    CASE 
        WHEN ins1.insurance_type = 1 THEN
            CASE 
                WHEN ins1.group_insurance = 1 THEN 'Group'
                WHEN ins1.group_insurance = 0 THEN 'Individual'
            END
        WHEN ins1.insurance_type = 2 THEN 'Group-Individual'
        ELSE '-'
    END AS TypeofInsuranceContract,
    COALESCE(ins1.insurance_count, 0) AS ofInsuranceContract,
    prp.insurance_cat AS InsuranceTypeDescription
FROM owner_dedup
LEFT JOIN insurance_1 ins1 
    ON ins1.RealEstateKey = owner_dedup.PropertyThesisID
LEFT JOIN insurance_2 ins2 
    ON ins2.RealEstateKey = owner_dedup.PropertyThesisID
LEFT JOIN contract_insurance ci 
    ON ci.RealEstateKey = owner_dedup.PropertyThesisID
LEFT JOIN {TBL_PRPROPERTY} prp 
    ON prp.id = TRY_CAST(owner_dedup.PropertyThesisID AS INT)
    AND prp.BusinessDate = '{Businessdate}'
""")

## Step 6: Final output — Write to delivery table
`SELECT DISTINCT` from `data_main` and write to `{env}-cdh-delivery.reo_cos_insurance.reo_cos_insurance`.

In [0]:
reoco_ins = spark.sql("""
SELECT DISTINCT
    OwnerName,
    PropertyThesisID,
    PropertyStatus,
    PropertyId,
    DWHPropertyID,
    PropertyCategory,
    PropertySubCategory,
    PropertyDescription,
    Address, Area, City, ZipCode,
    LandArea, NetArea,
    ContructionYear, MarketValue,
    InsuranceFlag,
    `Contract 1`  AS Contract_1,   `Start Date 1` AS Start_Date_1,  `End Date 1` AS End_Date_1,
    `Contract 2`  AS Contract_2,   `Start Date 2` AS Start_Date_2,  `End Date 2` AS End_Date_2,
    `Contract 3`  AS Contract_3,   `Start Date 3` AS Start_Date_3,  `End Date 3` AS End_Date_3,
    InsuranceCompany,
    InsuranceTypeDescription,
    InsuranceDateStart,
    InsuranceDateEnd,
    TypeofInsuranceContract,
    ofInsuranceContract
FROM data_main
""")

# Write to delivery table
reoco_ins.write.mode("overwrite").saveAsTable(DELIVERY_PATH)
print(f"Written {reoco_ins.count():,} rows to {DELIVERY_PATH}")